OUTLIER DETECTION USING Z-SCORE AND INTERQUARTILE RANGE

In [133]:
import mysql.connector
import configparser
import pandas as pd
from scipy.stats import iqr
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
#Location of the ini file
config = configparser.ConfigParser()
config.read('C:\\Users\\USER\\.spyder-py3\\erfpPROD.ini')

In [134]:
host=config['erfpPROD']['host']
user=config['erfpPROD']['user']
pwd=config['erfpPROD']['pwd']
database=config['erfpPROD']['database']

In [135]:
conn = mysql.connector.connect(
          host=host,
          user=user,
          passwd=pwd,
          database=database)
cursor = conn.cursor()

In [136]:
import_query_location = 'C:\\Users\\USER\\Documents\\Green_score\\NORMALISED\\CSR_WASTE.sql' 
import_query_open = open(import_query_location, 'r', encoding="utf8")
import_query_read = import_query_open.read()

In [137]:
sql_select_query = """SELECT 
        H.HOTEL_ID_VALUE AS HOTEL_ID
       ,H.HOTEL_NAME_NAME AS HOTEL_NAME   
       ,MEAS_ENERGY
       ,MEAS_ENERGY_AMT
       ,MEAS_ENERGY_UOM
       ,MEAS_ENERGY_PD 
       FROM GBTA_HOTEL_RFP.GBTA_ANSWERS_FOR_OFFER ANS
JOIN SOURCING.BI_ERFP_RFPENTITY RFP ON RFP.ID_VALUE = ANS.RFP_ID
JOIN SOURCING.BI_ERFP_HOTEL_FOR_RFPENTITY H ON RFP.ID_VALUE = H.OR_RFP_VALUE AND H.HOTEL_ID_VALUE = ANS.HOTEL_ID
WHERE 
      MEAS_ENERGY IS TRUE
   AND (MEAS_ENERGY_AMT IS NOT NULL OR MEAS_ENERGY_AMT <> 0)
   AND (MEAS_ENERGY_PD IS NOT NULL OR MEAS_ENERGY_PD <> 0) 
   AND MEAS_ENERGY_UOM LIKE 'BTU per Square Foot'"""

cursor.execute(import_query_read)

QUERY = cursor.fetchall()
print('Total Row(s):', cursor.rowcount)

# Importing data into a DataFrame
import pandas as pd
erfp_df = pd.DataFrame()
a=[]
for row in QUERY:
        a.append(row)

erfp_df = pd.DataFrame(a)
df_col_names =  [i[0] for i in cursor.description]
erfp_df.columns = df_col_names

In [138]:
erfp_df.dtypes

In [139]:
erfp_df.MEAS_WASTE_PCT = pd.to_numeric(erfp_df.MEAS_WASTE_PCT)
erfp_df.dtypes

In [140]:
waste_summary = pd.DataFrame({'MIN': erfp_df.MEAS_WASTE_PCT.min(), 
                               'MAX': erfp_df.MEAS_WASTE_PCT.max(), 
                               'AVG': erfp_df.MEAS_WASTE_PCT.mean(), 
                               'MEDIAN': erfp_df.MEAS_WASTE_PCT.median(), 
                               'MAD': erfp_df.MEAS_WASTE_PCT.mad()}, index=[0])
waste_summary

In [141]:
#MEAS_ENERGY_AMT = erfp_df.MEAS_ENERGY_AMT.to_numpy()
i = 'MEAS_WASTE_PCT'
# erfp_df[i].hist(color='slategray')
# plt.title("Waste diversion distribution", y=1.015, fontsize=22)
# plt.xlabel("Z-Score", labelpad=14)
# plt.ylabel(i, labelpad=14)

# Change line width
sns.violinplot( y=erfp_df.MEAS_WASTE_PCT, linewidth=5, inner='quartile')
#sns.plt.show()
sns.swarmplot(x=erfp_df.MEAS_WASTE_PCT)


In [142]:
n = len(erfp_df.MEAS_WASTE_PCT)
R = erfp_df.MEAS_WASTE_PCT.max() - erfp_df.MEAS_WASTE_PCT.min()
#  =  √n
# Width of intervals =  Range / (# of intervals)
n,R
sns.violinplot( y=erfp_df.MEAS_WASTE_PCT, linewidth=5, inner='point', split=1)

In [143]:
# Visualising to FIND the outliers

i = 'MEAS_WASTE_PCT'
 
plt.figure(figsize=(10,8))
plt.subplot(211)
plt.xlim(erfp_df[i].min()*1.5, erfp_df[i].max()*1.1)
plt.title("Kernel Distribution - WASTE amount (line graph)") 
ax = erfp_df[i].plot(kind='kde').margins(0.1)
 
plt.subplot(212)
plt.xlim(erfp_df[i].min()*1.5, erfp_df[i].max()*1.1)
plt.title("Kernel Distribution - WASTE amount (box plot)") 
sns.boxplot(x=erfp_df[i])

In [144]:
# Remove any zeros (otherwise we get (-inf)
erfp_df.loc[erfp_df.MEAS_WASTE_PCT == 0, 'MEAS_WASTE_PCT'] = np.nan
erfp_df.loc[erfp_df.MEAS_WASTE_PCT == 999, 'MEAS_WASTE_PCT'] = np.nan
erfp_df.loc[erfp_df.MEAS_WASTE_PCT == 9999, 'MEAS_WASTE_PCT'] = np.nan
erfp_df.loc[erfp_df.MEAS_WASTE_PCT == 9999.99, 'MEAS_WASTE_PCT'] = np.nan
 
# Drop NA
erfp_df.dropna(inplace=True)

erfp_df.shape

In [145]:
# Visualising Log Transform
erfp_df['Log_' + i] = np.log(erfp_df[i])

i = 'Log_MEAS_WASTE_PCT'
 
plt.figure(figsize=(10,8))
plt.subplot(211)
plt.xlim(erfp_df[i].min()*1.1, erfp_df[i].max()*1.1)
plt.title("Kernel Distribution of Log transfomed values WASTE amount (line graph)") 

ax = erfp_df[i].plot(kind='kde')
 
plt.subplot(212)
plt.xlim(erfp_df[i].min()*1.1, erfp_df[i].max()*1.1)
sns.boxplot(x=erfp_df[i])
plt.title("Kernel Distribution of Log transfomed values WASTE amount range=[0, 7900], ")

In [146]:
# OUTLIER DETECTION USING Z -SCORE
i = 'MEAS_WASTE_PCT'
_mean = erfp_df[i].mean()
_std = erfp_df[i].std()

erfp_df[i + '_Zscore'] = (erfp_df[i] - _mean)/(_std)
erfp_df[i + '_Zscore_abs'] = erfp_df[i + '_Zscore'].apply(np.abs)

In [147]:
erfp_df[i + '_Zscore'].hist(color='slategray')
plt.title("Standard Normal Distribution fo Z-Score - ENERGY amount", y=1.015, fontsize=22)
plt.xlabel("z-score", labelpad=14)
plt.ylabel(i, labelpad=14)

In [148]:
erfp_df['MEAS_WASTE_PCT_Zscore_abs'].min()

In [149]:
# erfp_df['MEAS_WASTE_PCT_Zscore_abs']
# # Visualising Log Transform
# erfp_df['Log_' + i] = np.log(erfp_df[i])

i = 'MEAS_WASTE_PCT_Zscore_abs'
 
plt.figure(figsize=(10,8))
plt.subplot(211)
plt.xlim(erfp_df[i].min()*1.1, erfp_df[i].max()*1.1)
 
ax = erfp_df[i].plot(kind='kde')

In [150]:
erfp_df.filter(erfp_df.Outlier == 0).describe()

In [ ]:
fig, ax = plt.subplots(figsize=(16,8))
ax.scatter(erfp_df['MEAS_WASTE_PCT'], erfp_df['MEAS_WASTE_PCT_Zscore'])
ax.set_xlabel('ENERGY_AMOUNT')
ax.set_ylabel('z_score')
plt.title("Z-score vs ENERGY amount - ENERGY amount (scatter plot)") 
plt.show()

In [151]:
# IQR
q1, q3= np.percentile(erfp_df.Log_MEAS_WASTE_PCT,[25,75])
iqr = q3 - q1
iqr, q1, q3, 1.5*iqr

In [152]:
LB = q1 -(1.5 * iqr) 
UB = q3 +(1.5 * iqr) 
LB,UB

In [153]:
erfp_df.loc[erfp_df['Log_MEAS_WASTE_PCT'] > UB]

In [154]:
i = 'Log_MEAS_WASTE_PCT'

plt.figure(figsize=(10,8))
plt.subplot(211)
plt.xlim(erfp_df[i].min(), erfp_df[i].max()*1.1)
plt.axvline(x=LB, color='r', ls = '--')
plt.axvspan(xmin=LB, xmax=erfp_df[i].min(), facecolor='#2ca02c', alpha=0.5)
plt.axvline(x=UB, color='r', ls = '--')
plt.axvspan(xmin=UB, xmax=erfp_df[i].max()*1.1, facecolor='#2ca02c', alpha=0.5)
plt.title("IQR Outlier detection - WASTE amount (line graph)") 

ax = erfp_df[i].plot(kind='kde')

plt.subplot(212)
plt.xlim(erfp_df[i].min()*1.1, erfp_df[i].max()*1.1)
sns.boxplot(x=erfp_df[i])
plt.axvline(x=LB, color='r', ls = '--')
plt.axvspan(xmin=LB, xmax=erfp_df[i].min()*1.1, facecolor='#2ca02c', alpha=0.5)
plt.axvline(x=UB, color='r', ls = '--')
plt.axvspan(xmin=UB, xmax=erfp_df[i].max()*1.1, facecolor='#2ca02c', alpha=0.5)
plt.title("IQR Outlier detection - WASTE amount (box plot)") 

In [112]:
i = 'Log_MEAS_WASTE_PCT'

erfp_df['Outlier'] = 0
 
erfp_df.loc[erfp_df[i] < LB, 'Outlier'] = 1
erfp_df.loc[erfp_df[i] > UB, 'Outlier'] = 1

In [113]:
erfp_df.MEAS_WASTE_PCT.mad()

In [114]:
erfp_df.shape

In [115]:
erfp_df[erfp_df.Outlier != 1].MEAS_WASTE_PCT.max()

In [116]:
erfp_df[erfp_df.Outlier == 1].count()

In [80]:
# OUTLIER
erfp_df[erfp_df.Outlier == 1].to_excel('C:\\Users\\USER\\Documents\\Green_score\\ENERGY_outliers.xlsx', 
                sheet_name='ENERGY_outlier')

In [117]:
import datetime as d
file = "C:\\Users\\USER\\Documents\\Green_score\\WASTE_CERTIFIED.xlsx" 
with pd.ExcelWriter(file) as writer: 
    erfp_df[erfp_df.Outlier == 1].to_excel(writer, sheet_name='OUTLIER', header=True, encoding='utf-8', index=False, freeze_panes=(1,1))
    erfp_df[erfp_df.Outlier == 0].to_excel(writer, sheet_name='BENCHMARK', header=True, encoding='utf-8', index=False, freeze_panes=(1,1))

In [127]:
erfp_df[erfp_df['Outlier']==0].quantile(np.linspace(.1, 1, 9, 0))